# AlphaLOB Phase 2 — Notebook 05: Institutional-Grade Walk-Forward Backtest

**Inputs:**
- `/content/drive/MyDrive/AlphaLOB/lob_features.parquet` (from Notebook 02)
- `/content/drive/MyDrive/AlphaLOB/lobster_transformer.onnx` (from Notebook 03)
- `/content/drive/MyDrive/AlphaLOB/regime_hmm.pkl` (from Notebook 04)

**Outputs:** Metrics table + equity curve PNG + walk-forward comparison chart

## Walk-Forward Structure (3 OOS Windows + Final OOS)
```
Window 1: Train [0%→50%]  Test [50%→60%]
Window 2: Train [0%→60%]  Test [60%→70%]
Window 3: Train [0%→70%]  Test [70%→80%]
Final OOS: [80%→100%]  ← NEVER TOUCHED until final evaluation
```

## Critical Requirements (from audit)
1. `build_lob_tensor()` must produce exactly `(batch, 10, 4)` with correct channels
2. NaN to 0.0 via `np.nan_to_num()` BEFORE every ONNX inference call
3. STRICTLY CAUSAL: at tick `i`, only use data available at tick `i`
4. `daily_vol` computed on TRAINING data only (point-in-time) — never test data
5. All walk-forward window boundaries are exclusive of the Final OOS [80%→100%]
6. All paths use `/content/drive/MyDrive/AlphaLOB/`

## Metrics Reported
- Sharpe, Sortino, Calmar, Omega ratios
- Max Drawdown (fraction and duration)
- 30s Directional Accuracy (key performance metric)
- Break-Even Transaction Cost (bps)
- Win Rate, Profit Factor

---


In [ ]:
# Cell 1: Mount Google Drive + Install dependencies
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

!pip install polars pyarrow onnxruntime hmmlearn joblib matplotlib --quiet
print('✅ Dependencies installed')


In [ ]:
# Cell 2: Imports and configuration

import numpy as np
import polars as pl
import onnxruntime as ort
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ── Paths (ALL on Google Drive as required) ────────────────────────────────
PARQUET_IN = '/content/drive/MyDrive/AlphaLOB/lob_features.parquet'
ONNX_PATH  = '/content/drive/MyDrive/AlphaLOB/lobster_transformer.onnx'
HMM_PATH   = '/content/drive/MyDrive/AlphaLOB/regime_hmm.pkl'

# Verify all inputs exist before proceeding
for path, name in [(PARQUET_IN, 'lob_features.parquet'),
                   (ONNX_PATH,  'lobster_transformer.onnx'),
                   (HMM_PATH,   'regime_hmm.pkl')]:
    assert os.path.exists(path), f'{name} not found: {path}\nRun preceding notebooks first.'
    size_mb = os.path.getsize(path) / 1e6
    print(f'  ✅ {name} ({size_mb:.1f} MB)')

# ── Walk-forward windows (STRICTLY NON-OVERLAPPING test periods) ────────────
# Train windows always extend from 0 (increasing anchor) — this is expanding-window WF.
# Test windows are exclusive: each window's test data is never used for another test.
# Final OOS is [80%→100%] and MUST NOT be touched until Cell 11.
WINDOWS = [
    {'name': 'Window 1', 'train_end': 0.50, 'test_start': 0.50, 'test_end': 0.60},
    {'name': 'Window 2', 'train_end': 0.60, 'test_start': 0.60, 'test_end': 0.70},
    {'name': 'Window 3', 'train_end': 0.70, 'test_start': 0.70, 'test_end': 0.80},
]
FINAL_OOS_START = 0.80   # NEVER TOUCH until Cell 11

# ── Strategy parameters ────────────────────────────────────────────────────
LONG_THRESHOLD  = 0.60   # Enter LONG if P(UP) > 0.60
SHORT_THRESHOLD = 0.40   # Enter SHORT if P(UP) < 0.40
KELLY_CAP       = 0.25   # Maximum Kelly fraction (25% of capital)
STOP_LOSS_FRAC  = -0.005 # Stop out at -0.5% raw return per trade
INITIAL_CAPITAL = 100_000.0
HORIZON_TICKS   = 300    # 30s at 10 ticks/sec

# ── Transaction cost model ─────────────────────────────────────────────────
SLIPPAGE_MIN_BPS = 0.5
SLIPPAGE_MAX_BPS = 2.0
ADV_FRAC         = 0.01  # we trade 1% of ADV

print()
print('✅ Configuration loaded')
print(f'   Walk-forward windows: {len(WINDOWS)}')
print(f'   Final OOS reserved:   {int(FINAL_OOS_START*100)}%–100%')
print(f'   Long threshold:       {LONG_THRESHOLD}')
print(f'   Short threshold:      {SHORT_THRESHOLD}')
print(f'   Kelly cap:            {KELLY_CAP*100:.0f}%')
print(f'   Horizon:              {HORIZON_TICKS} ticks (30s)')


In [ ]:
# Cell 3: Load all data and models
#
# DATA LOADING:
#   fill_null(0.0).fill_nan(0.0) applied immediately on load.
#   This ensures ONNX inference never sees NaN inputs.
#   (Defense-in-depth: np.nan_to_num() is ALSO applied per-batch in build_lob_tensor)
#
# ONNX SESSION:
#   CPUExecutionProvider is mandatory — T4 GPU providers are not available for
#   onnxruntime CPU builds. Latency target: p99 < 15ms per batch.
#
# HMM MODEL:
#   regime_names dict is attached to model object (set in NB04).
#   If AttributeError on model.regime_names → re-run Notebook 04.

print('=' * 62)
print('  Cell 3: Load Data and Models')
print('=' * 62)

print('\n[1] Loading feature data...')
t0 = time.time()
df = pl.read_parquet(PARQUET_IN)

# MANDATORY: purge any residual NaN/null before any computations
df = df.fill_null(0.0).fill_nan(0.0)
n_total = len(df)
print(f'  Loaded {n_total:,} rows in {time.time()-t0:.1f}s')
print(f'  Columns: {len(df.columns)}')

# Verify required columns for build_lob_tensor()
required_cols = (
    ['mid_price', 'wofi_z', 'kyle_lambda_z', 'realized_vol', 'autocorrelation'] +
    [f'bid_price_{l}' for l in range(10)] + [f'ask_price_{l}' for l in range(10)] +
    [f'bid_vol_{l}'  for l in range(10)] + [f'ask_vol_{l}'  for l in range(10)]
)
missing = [c for c in required_cols if c not in df.columns]

if missing:
    print(f'\n  ⚠️  Missing columns: {missing}')
    print(f'  Attempting to merge raw LOB columns from lob_data.parquet...')
    raw_path = PARQUET_IN.replace('lob_features.parquet', 'lob_data.parquet')
    if os.path.exists(raw_path):
        df_raw  = pl.read_parquet(raw_path).head(n_total)
        df_raw  = df_raw.fill_null(0.0).fill_nan(0.0)
        raw_lob_cols = [c for c in df_raw.columns
                        if (c.startswith('bid_') or c.startswith('ask_'))
                        and c not in df.columns]
        df = df.hstack(df_raw.select(raw_lob_cols))
        print(f'  ✅ Merged {len(raw_lob_cols)} raw LOB columns')
    else:
        raise FileNotFoundError(
            f'Raw LOB data not found at {raw_path}. '
            f'Ensure Notebook 01 saved lob_data.parquet to Drive.'
        )
else:
    print(f'  ✅ All required columns present')

# Ensure spread_z exists (needed by LOBDataset in NB03 but not strictly needed here)
if 'spread_z' not in df.columns:
    df = df.with_columns(pl.lit(0.0).alias('spread_z'))

print('\n[2] Loading ONNX session...')
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
onnx_inputs  = [i.name for i in sess.get_inputs()]
onnx_outputs = [o.name for o in sess.get_outputs()]
print(f'  ✅ ONNX loaded | Inputs: {onnx_inputs} | Outputs: {onnx_outputs}')

print('\n[3] Loading RegimeHMM...')
hmm_model = joblib.load(HMM_PATH)
assert hasattr(hmm_model, 'regime_names'), (
    'regime_names attribute missing from HMM model!\n'
    'Re-run Notebook 04 to regenerate regime_hmm.pkl with regime_names attached.'
)
regime_names = hmm_model.regime_names
print(f'  ✅ HMM loaded | regime_names: {regime_names}')
print(f'  States: {hmm_model.n_components} | Cov type: {hmm_model.covariance_type}')

print(f'\n✅ All inputs loaded. n_total = {n_total:,} ticks')


In [ ]:
# Cell 4: Build LOB tensor and inference helpers
#
# CRITICAL: build_lob_tensor() MUST construct exactly (batch, 10, 4) where
# the 4 channels per level are:
#   Channel 0: normalized_price_dist = ((bid_p + ask_p) / 2 - mid) / (mid + 1e-9)
#              Captures how far this level's midpoint is from the best mid.
#              Units: fraction (dimensionless). Typical range: [-0.001, 0.0].
#   Channel 1: log_normalized_vol    = log1p(bid_vol + ask_vol)
#              Log-transform handles the heavy-tailed log-normal volume distribution.
#              Always non-negative.
#   Channel 2: wofi_z                = rolling Z-score of WOFI (from Notebook 02)
#              Shared across all 10 levels (same tick-level feature for each level).
#              Range: approximately [-5, 5] (clipped in rolling_zscore).
#   Channel 3: kyle_lambda_z         = rolling Z-score of Kyle's Lambda
#              Shared across all 10 levels (tick-level price impact feature).
#              Range: approximately [-5, 5].
#
# WHY SAME WOFI/KYLE FOR ALL LEVELS?
#   WOFI and Kyle's lambda are TICK-LEVEL features — computed from the full order book.
#   They don't vary level-by-level. Repeating them across levels lets the Transformer
#   attend to the full-book signal while processing each level's price/volume features.
#   This matches the LOBDataset in Notebook 03.
#
# MANDATORY NaN SANITIZATION:
#   np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0) is applied AFTER building X.
#   This handles: log1p(negative) → NaN, division by near-zero mid → Inf.
#   Without this, ONNX inference produces NaN outputs → NaN probabilities → wrong signals.

print('=' * 62)
print('  Cell 4: Define LOB Tensor and Inference Functions')
print('=' * 62)

def build_lob_tensor(df_slice: pl.DataFrame) -> np.ndarray:
    """
    Build (n, 10, 4) float32 input tensor for ONNX inference.
    Channels: [normalized_price_dist, log_normalized_vol, wofi_z, kyle_lambda_z]

    STRICTLY CAUSAL: uses only df_slice[i] data at each tick i.
    No rolling windows computed here — those are pre-computed in Notebook 02.
    """
    n        = len(df_slice)
    N_LEVELS = 10
    X        = np.zeros((n, N_LEVELS, 4), dtype=np.float32)

    mid  = df_slice['mid_price'].to_numpy().astype(np.float32)
    wofi = df_slice['wofi_z'].to_numpy().astype(np.float32)
    kyle = df_slice['kyle_lambda_z'].to_numpy().astype(np.float32)

    for lvl in range(N_LEVELS):
        bp = df_slice[f'bid_price_{lvl}'].to_numpy().astype(np.float32)
        ap = df_slice[f'ask_price_{lvl}'].to_numpy().astype(np.float32)
        bv = df_slice[f'bid_vol_{lvl}'].to_numpy().astype(np.float32)
        av = df_slice[f'ask_vol_{lvl}'].to_numpy().astype(np.float32)

        # Channel 0: normalized price distance from mid
        level_mid   = (bp + ap) * 0.5
        price_dist  = (level_mid - mid) / (mid + 1e-9)

        # Channel 1: log-normalized combined volume
        vol_total   = np.log1p(np.maximum(bv + av, 0.0))

        # Channels 2, 3: tick-level features (same value repeated for all levels)
        X[:, lvl, 0] = price_dist
        X[:, lvl, 1] = vol_total
        X[:, lvl, 2] = wofi
        X[:, lvl, 3] = kyle

    # MANDATORY: sanitize NaN/Inf before ONNX inference
    # Without this: NaN inputs → NaN probabilities → random signals
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X


def run_inference_batch(X: np.ndarray, batch_size: int = 2048) -> np.ndarray:
    """
    Run ONNX inference in batches. Returns P(UP) for 30s horizon.
    X shape: (n, 10, 4)
    Returns: (n,) float32 array of probabilities that price goes UP in 30s
    """
    all_probs = []
    for i in range(0, len(X), batch_size):
        batch   = X[i:i + batch_size]
        outputs = sess.run(['dir_30s'], {'lob_snapshot': batch})
        # outputs[0] shape: (batch, 2) = [P(DOWN), P(UP)] (softmax)
        all_probs.append(outputs[0][:, 1])
    return np.concatenate(all_probs)


def get_regimes(df_slice: pl.DataFrame) -> np.ndarray:
    """
    Predict HMM regimes. Returns string array of regime names.
    Input: realized_vol, autocorrelation columns.
    """
    rv   = df_slice['realized_vol'].to_numpy().astype(np.float64)
    ac   = df_slice['autocorrelation'].to_numpy().astype(np.float64)
    X_hmm = np.column_stack([rv, ac])
    # Sanitize before HMM — same reason as hmmlearn training
    X_hmm = np.nan_to_num(X_hmm, nan=0.0, posinf=0.0, neginf=0.0)
    states = hmm_model.predict(X_hmm)
    return np.array([regime_names[int(s)] for s in states])


# ── Quick sanity check: build tensor on first 100 rows ─────────────────────
print('\n[1] Sanity check: build_lob_tensor on first 100 rows...')
X_test_build = build_lob_tensor(df.head(100))
assert X_test_build.shape == (100, 10, 4), f'Wrong shape: {X_test_build.shape}'
assert not np.isnan(X_test_build).any(), 'NaN in tensor output!'
assert not np.isinf(X_test_build).any(), 'Inf in tensor output!'
print(f'  ✅ Shape: {X_test_build.shape}  (correct: (100, 10, 4))')
print(f'  ✅ No NaN/Inf in tensor')

print('\n[2] Sanity check: ONNX inference on dummy batch...')
dummy_probs = run_inference_batch(X_test_build)
assert dummy_probs.shape == (100,), f'Wrong probs shape: {dummy_probs.shape}'
assert (0 <= dummy_probs).all() and (dummy_probs <= 1).all(), 'Probs not in [0,1]!'
print(f'  ✅ ONNX inference OK | probs shape: {dummy_probs.shape}')
print(f'  ✅ Probs range: [{dummy_probs.min():.4f}, {dummy_probs.max():.4f}]')

print('\n[3] Sanity check: HMM regime prediction on first 100 rows...')
regimes_test = get_regimes(df.head(100))
assert regimes_test.shape == (100,), f'Wrong regimes shape: {regimes_test.shape}'
unique_regimes = set(regimes_test.tolist())
valid_regimes  = {'TRENDING', 'MEAN_REVERTING', 'VOLATILE'}
assert unique_regimes.issubset(valid_regimes), f'Unknown regime labels: {unique_regimes - valid_regimes}'
print(f'  ✅ Regime prediction OK | unique regimes seen: {unique_regimes}')

print('\n✅ All helper functions defined and verified')


In [ ]:
# Cell 5: Market impact model + Kelly criterion + Risk metrics
#
# MARKET IMPACT MODEL: Square-root model (industry standard at HFT firms)
#   impact(Q) = η · σ_daily · √(Q / ADV)
#   η: market impact coefficient (0.5 for liquid BTC markets)
#   σ_daily: POINT-IN-TIME daily vol from TRAINING data only
#   Q/ADV: fraction of average daily volume being traded
#
# KELLY CRITERION: f* = (p·b - q) / b
#   p = probability of winning (from ONNX model)
#   q = 1 - p
#   b = win_fraction / loss_fraction (assumed 1:1 for simplicity)
#   Capped at KELLY_CAP (25%) to avoid over-sizing in high-confidence signals
#
# DAILY VOL COMPUTATION:
#   MUST be computed on TRAINING data only.
#   Using test data vol = look-ahead bias (you know the future vol at entry time).
#   Formula: std(log_returns) × √(ticks_per_day)
#   ticks_per_day = 10 ticks/sec × 3600s/hr × 24hr = 864,000

def compute_sqrt_market_impact(q_frac: float, daily_vol: float) -> float:
    """
    Square-root market impact in basis points.
    q_frac: fraction of ADV being traded (e.g., 0.01)
    daily_vol: realized daily volatility (e.g., 0.02 = 2%)
    Returns: total impact in bps
    """
    eta = 0.5   # market impact coefficient for liquid crypto
    impact_pct = eta * daily_vol * np.sqrt(q_frac)
    return impact_pct * 10_000   # bps


def kelly_fraction(prob_win: float) -> float:
    """
    Kelly criterion for binary payoff (equal win/loss).
    f* = p - q = 2p - 1  when b=1 (symmetric payoff)
    Capped at KELLY_CAP, floored at 0.
    """
    f = max(2 * prob_win - 1.0, 0.0)
    return min(f, KELLY_CAP)


def compute_metrics(trade_returns: np.ndarray) -> dict:
    """
    Compute institutional risk metrics from per-trade P&L array.
    All annualization uses trade frequency (not calendar time).
    trade_returns: 1D array of net returns per trade (fractions, e.g. 0.001 = 0.1%)
    """
    N_EMPTY = {
        'sharpe_ratio': np.nan, 'sortino_ratio': np.nan, 'calmar_ratio': np.nan,
        'omega_ratio': np.nan, 'max_drawdown': np.nan, 'max_dd_duration': 0,
        'total_return': 0.0, 'ann_return': np.nan, 'win_rate': np.nan,
        'profit_factor': np.nan, 'avg_trade_pnl': 0.0,
        'break_even_bps': 0.0, 'total_trades': 0,
    }
    if len(trade_returns) < 10:
        return N_EMPTY

    n = len(trade_returns)
    mean_ret = float(np.mean(trade_returns))
    std_ret  = float(np.std(trade_returns, ddof=1)) if n > 1 else 1e-9

    # Annualization: assume each trade = 30s horizon, ~1152 trades/day, ~290k/year
    # Use sqrt(n_trades_year) as annualization factor
    TICKS_PER_YEAR = 10 * 3600 * 24 * 365       # total ticks in a year
    TRADE_FREQ     = TICKS_PER_YEAR / HORIZON_TICKS  # expected trades/year if always in market
    ann_factor     = np.sqrt(TRADE_FREQ)

    # Sharpe Ratio (rf = 0)
    sharpe  = (mean_ret / std_ret) * ann_factor if std_ret > 1e-10 else 0.0

    # Sortino (downside vol only)
    down    = trade_returns[trade_returns < 0.0]
    down_std = float(np.std(down, ddof=1)) if len(down) > 1 else 1e-9
    sortino = (mean_ret / down_std) * ann_factor if down_std > 1e-10 else 0.0

    # Equity curve and max drawdown
    eq      = np.cumprod(1.0 + trade_returns)
    roll_max = np.maximum.accumulate(eq)
    dds      = (eq - roll_max) / roll_max
    max_dd   = float(abs(dds.min()))

    # Drawdown duration (in trades)
    in_dd  = dds < 0.0
    dd_lengths = []
    cnt = 0
    for x in in_dd:
        if x:
            cnt += 1
        else:
            if cnt > 0:
                dd_lengths.append(cnt)
            cnt = 0
    if cnt > 0:
        dd_lengths.append(cnt)
    max_dd_dur = int(max(dd_lengths)) if dd_lengths else 0

    # Annualized return
    total_ret = float(eq[-1] - 1.0)
    # Approximate annualization
    holding_frac = n / TRADE_FREQ   # fraction of year spent in positions
    ann_ret = (1.0 + total_ret) ** (1.0 / max(holding_frac, 1e-6)) - 1.0

    # Calmar
    calmar = ann_ret / max_dd if max_dd > 1e-9 else np.nan

    # Omega ratio
    gains  = float(np.sum(trade_returns[trade_returns > 0]))
    losses = float(abs(np.sum(trade_returns[trade_returns < 0])))
    omega  = gains / losses if losses > 1e-9 else np.nan

    # Win rate and profit factor
    win_rate       = float(np.mean(trade_returns > 0))
    gross_profit   = gains
    gross_loss     = losses
    profit_factor  = gross_profit / gross_loss if gross_loss > 1e-9 else np.nan

    # Break-even transaction cost (bps): the cost at which mean net return = 0
    # Each trade has 2 legs (entry + exit), so cost is per half-turn
    # break_even = mean gross return (before costs) expressed in bps
    break_even_bps = mean_ret * 10_000

    return {
        'sharpe_ratio':    sharpe,
        'sortino_ratio':   sortino,
        'calmar_ratio':    calmar,
        'omega_ratio':     omega,
        'max_drawdown':    max_dd,
        'max_dd_duration': max_dd_dur,
        'total_return':    total_ret,
        'ann_return':      ann_ret,
        'win_rate':        win_rate,
        'profit_factor':   profit_factor,
        'avg_trade_pnl':   mean_ret,
        'break_even_bps':  break_even_bps,
        'total_trades':    n,
    }

print('✅ Market impact, Kelly, and metrics functions defined')


In [ ]:
# Cell 6: Core backtesting engine
#
# STRICT CAUSALITY GUARANTEE:
#   At each tick i, we ONLY use:
#     - df[i]: current LOB snapshot features (pre-computed in NB02 from past data)
#     - ONNX model: trained on data BEFORE the test window
#     - HMM model: trained on X_train (0% to train_end%)
#     - daily_vol: computed from df[0:train_end_idx] ONLY
#   We NEVER look at df[i+1], df[i+2], etc. when making the signal at tick i.
#   The exit price (mid[i + HORIZON_TICKS]) is observed AFTER the trade is entered.
#   This is the correct causal setup — we enter at i and observe the outcome later.
#
# REGIME FILTERING:
#   In VOLATILE regime → skip trade (no edge, high cost of error)
#   In TRENDING or MEAN_REVERTING → trade if signal is strong enough
#
# TRADE ADVANCE:
#   After entering a trade at tick i, we advance by HORIZON_TICKS (300 ticks = 30s).
#   This prevents overlapping trades (same capital can't be in 2 trades at once).

def run_backtest_on_window(df_test_slice: pl.DataFrame,
                           daily_vol_train: float,
                           window_name: str) -> tuple:
    """
    Run full backtest on one OOS window.
    Returns: (metrics_dict, trade_returns_array)

    daily_vol_train: computed from TRAINING data only (point-in-time).
    """
    n = len(df_test_slice)
    if n < HORIZON_TICKS + 10:
        print(f'  ⚠️  {window_name}: test slice too small ({n} ticks), skipping')
        return compute_metrics(np.array([])), np.array([])

    print(f'  [{window_name}] n={n:,} ticks | daily_vol={daily_vol_train*100:.3f}%')

    # ── 1. Build all model inputs at once (batch inference) ─────────────────
    t0 = time.time()
    X = build_lob_tensor(df_test_slice)
    # MANDATORY NaN sanitization before ONNX (belt-and-suspenders)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # ── 2. Batch ONNX inference ──────────────────────────────────────────────
    prob_up = run_inference_batch(X)   # (n,) P(UP in 30s)

    # ── 3. Batch HMM regime prediction ──────────────────────────────────────
    regimes = get_regimes(df_test_slice)   # (n,) string array

    # ── 4. Mid prices for P&L computation ───────────────────────────────────
    mid = df_test_slice['mid_price'].to_numpy()

    # ── 5. Pre-compute transaction cost (deterministic for clarity) ──────────
    rng = np.random.default_rng(seed=42)
    impact_bps     = compute_sqrt_market_impact(ADV_FRAC, daily_vol_train)
    # Randomize slippage between min and max (simulates variable execution quality)
    slip_per_trade = rng.uniform(SLIPPAGE_MIN_BPS, SLIPPAGE_MAX_BPS, size=n)
    total_cost_pct_arr = (slip_per_trade + impact_bps) / 10_000   # per trade (one leg)

    # ── 6. Trade simulation ──────────────────────────────────────────────────
    trade_returns = []
    dir_correct   = 0
    dir_total     = 0

    i = 0
    while i < n - HORIZON_TICKS:
        # CAUSALITY CHECK: everything below uses index i or earlier
        p_up   = float(prob_up[i])
        regime = str(regimes[i])

        # Regime filter: skip VOLATILE (no edge, avoid large losses)
        if regime == 'VOLATILE':
            i += 1
            continue

        # Signal generation
        if p_up > LONG_THRESHOLD:
            direction = 1     # LONG: bet price goes UP
        elif p_up < SHORT_THRESHOLD:
            direction = -1    # SHORT: bet price goes DOWN
        else:
            i += 1
            continue          # No signal

        # Kelly position sizing
        prob_win = p_up if direction == 1 else (1.0 - p_up)
        f_kelly  = kelly_fraction(prob_win)
        if f_kelly < 0.01:
            i += 1
            continue           # Signal too weak (edge < 1%)

        # Entry and exit prices (CAUSAL: exit observed after HORIZON_TICKS)
        entry_price = float(mid[i])
        exit_price  = float(mid[i + HORIZON_TICKS])

        if entry_price < 1e-9:
            i += 1
            continue

        # Raw fractional return
        raw_ret = direction * (exit_price - entry_price) / entry_price

        # Apply stop-loss (hard cap on downside)
        raw_ret = max(raw_ret, STOP_LOSS_FRAC)

        # Net return after transaction costs (round-trip: 2 × one-way cost)
        total_cost = 2.0 * float(total_cost_pct_arr[i])
        net_ret    = f_kelly * (raw_ret - total_cost)

        trade_returns.append(net_ret)

        # Directional accuracy tracking (actual vs predicted direction)
        actual_dir = 1 if exit_price > entry_price else -1
        if direction == actual_dir:
            dir_correct += 1
        dir_total += 1

        # Advance by HORIZON_TICKS to avoid overlapping trades
        i += HORIZON_TICKS

    elapsed = time.time() - t0
    trade_arr     = np.array(trade_returns, dtype=np.float64)
    dir_acc       = dir_correct / max(dir_total, 1)
    metrics       = compute_metrics(trade_arr)
    metrics['directional_acc_30s'] = dir_acc
    metrics['window']  = window_name
    metrics['elapsed_s'] = elapsed

    print(f'       Trades={len(trade_arr)} | Dir.Acc={dir_acc*100:.1f}% | '
          f'Sharpe={metrics["sharpe_ratio"]:.3f} | MaxDD={metrics["max_drawdown"]*100:.1f}%')

    return metrics, trade_arr


print('✅ Backtest engine defined')
print(f'   Horizon: {HORIZON_TICKS} ticks (30s)')
print(f'   Regime filter: VOLATILE ticks excluded')
print(f'   Stop-loss: {STOP_LOSS_FRAC*100:.1f}%')
print(f'   Kelly cap: {KELLY_CAP*100:.0f}%')


In [ ]:
# Cell 7: Run all 3 walk-forward windows
#
# DAILY VOL COMPUTATION — POINT-IN-TIME (CRITICAL):
#   We compute daily_vol using ONLY the training slice [0 → train_end_idx].
#   Formula: std(log_returns_train) × √(10 × 3600 × 24)
#   = per-tick std × √(ticks per day)
#   This gives approximate daily fractional volatility.
#
#   If we used test data vol: we'd know the future vol at trade time → look-ahead.
#   If we used the full dataset vol: same issue (test data contributes).
#
# WALK-FORWARD ISOLATION:
#   Each window's test slice is completely independent of other test slices.
#   The train slice grows (expanding window), simulating real incremental data.
#   The three test windows cover [50%,80%] in total.
#   [80%,100%] is NEVER touched — reserved for Cell 11 final OOS.

print('=' * 62)
print('  Cell 7: Walk-Forward Backtest — 3 Windows')
print('=' * 62)
print()
print('  Window structure (expanding-window walk-forward):')
for w in WINDOWS:
    te = int(n_total * w['test_end'])
    ts = int(n_total * w['test_start'])
    tr = int(n_total * w['train_end'])
    print(f'    {w["name"]}: Train[0,{tr:,}] Test[{ts:,},{te:,}] ({te-ts:,} test ticks)')
print()

all_metrics = []
all_returns = []

for window in WINDOWS:
    # Slice indices
    train_end_idx  = int(n_total * window['train_end'])
    test_start_idx = int(n_total * window['test_start'])
    test_end_idx   = int(n_total * window['test_end'])

    # Data slices
    df_train = df.slice(0, train_end_idx)
    df_test  = df.slice(test_start_idx, test_end_idx - test_start_idx)

    # POINT-IN-TIME daily vol: TRAINING data only
    train_mid  = df_train['mid_price'].to_numpy()
    train_lret = np.diff(np.log(np.maximum(train_mid, 1e-6)))
    # Remove any NaN from log-diff computation
    train_lret = train_lret[np.isfinite(train_lret)]
    # Daily vol = per-tick std × sqrt(ticks per day)
    TICKS_PER_DAY = 10 * 3600 * 24   # 10 ticks/sec × 86400s/day
    daily_vol = float(np.std(train_lret)) * np.sqrt(TICKS_PER_DAY)

    print(f'  {window["name"]}:')
    print(f'    Train ticks: {len(df_train):,} | Test ticks: {len(df_test):,}')
    print(f'    Point-in-time daily_vol: {daily_vol*100:.4f}%')

    metrics, returns = run_backtest_on_window(df_test, daily_vol, window['name'])
    all_metrics.append(metrics)
    all_returns.append(returns)
    print()

print('✅ All 3 walk-forward windows complete')
mean_sharpe = float(np.nanmean([m['sharpe_ratio'] for m in all_metrics]))
mean_acc    = float(np.nanmean([m['directional_acc_30s'] for m in all_metrics]))
print(f'   Mean Sharpe (3 windows): {mean_sharpe:.3f}')
print(f'   Mean 30s Dir. Accuracy:  {mean_acc*100:.2f}%')


In [ ]:
# Cell 8: Print full metrics table

print('=' * 75)
print('  WALK-FORWARD BACKTEST — FULL RESULTS TABLE')
print('=' * 75)

METRIC_ROWS = [
    ('sharpe_ratio',        'Sharpe Ratio',             '2.3',    '{:.3f}'),
    ('sortino_ratio',       'Sortino Ratio',             '3.1',    '{:.3f}'),
    ('calmar_ratio',        'Calmar Ratio',              '>1.0',   '{:.3f}'),
    ('omega_ratio',         'Omega Ratio',               '>1.0',   '{:.3f}'),
    ('max_drawdown',        'Max Drawdown',              '<8.3%',  '{:.2%}'),
    ('max_dd_duration',     'Max DD Duration (trades)',  '—',      '{:.0f}'),
    ('total_return',        'Total OOS Return',          '—',      '{:.2%}'),
    ('win_rate',            'Win Rate',                  '—',      '{:.2%}'),
    ('profit_factor',       'Profit Factor',             '>1.5',   '{:.3f}'),
    ('avg_trade_pnl',       'Avg Trade P&L',             '—',      '{:.6f}'),
    ('break_even_bps',      'Break-Even Cost (bps)',     '8.2',    '{:.2f}'),
    ('total_trades',        'Total Trades',              '—',      '{:.0f}'),
    ('directional_acc_30s', '30s Dir. Accuracy',         '58.2%',  '{:.2%}'),
]

# Print header
col_w = 14
header = f'  {"Metric":<32} {"Target":>7}'
for m in all_metrics:
    header += f'  {m["window"]:>{col_w}}'
header += f'  {"MEAN":>{col_w}}'
print(header)
print('  ' + '-' * (32 + 7 + (col_w + 2) * (len(all_metrics) + 1) + 4))

# Print each metric row
for key, label, target, fmt in METRIC_ROWS:
    row = f'  {label:<32} {target:>7}'
    vals = []
    for m in all_metrics:
        v = m.get(key, np.nan)
        try:
            row += f'  {fmt.format(v):>{col_w}}'
        except (ValueError, TypeError):
            row += f'  {"nan":>{col_w}}'
        vals.append(v if np.isfinite(v) else np.nan)
    mean_v = np.nanmean(vals) if any(np.isfinite(v) for v in vals) else np.nan
    try:
        row += f'  {fmt.format(mean_v):>{col_w}}'
    except (ValueError, TypeError):
        row += f'  {"nan":>{col_w}}'
    print(row)

print('=' * 75)
mean_sharpe = float(np.nanmean([m['sharpe_ratio'] for m in all_metrics]))
mean_sortino = float(np.nanmean([m['sortino_ratio'] for m in all_metrics]))
mean_acc    = float(np.nanmean([m['directional_acc_30s'] for m in all_metrics]))
mean_be     = float(np.nanmean([m['break_even_bps'] for m in all_metrics]))
mean_dd     = float(np.nanmean([m['max_drawdown'] for m in all_metrics]))

print(f'\n  ★  Mean Sharpe  (3 OOS windows): {mean_sharpe:.3f}   (target: 2.3)')
print(f'  ★  Mean Sortino (3 OOS windows): {mean_sortino:.3f}   (target: 3.1)')
print(f'  ★  Mean 30s Dir. Accuracy:       {mean_acc*100:.2f}%   (target: 58.2%)')
print(f'  ★  Mean Break-even Cost:         {mean_be:.2f} bps     (target: 8.2 bps)')
print(f'  ★  Mean Max Drawdown:            {mean_dd*100:.2f}%')

# Interpretation
print()
if mean_acc >= 0.55:
    print('  ✅ Model shows genuine predictive edge (>55% 30s accuracy)')
elif mean_acc >= 0.52:
    print('  ⚠️  Modest edge. Consider: more data, more epochs, or larger model.')
else:
    print('  ❌ Near-random. Check: Notebook 01 volume fix, NaN purge in NB02, training in NB03.')

if mean_sharpe >= 1.5:
    print('  ✅ Sharpe > 1.5 — institutionally viable walk-forward performance')
elif mean_sharpe >= 0.5:
    print('  ⚠️  Sharpe 0.5–1.5 — some edge but below institutional threshold')
else:
    print('  ❌ Sharpe < 0.5 — transaction costs likely exceed the signal edge')


In [ ]:
# Cell 9: Plot equity curves and walk-forward comparison charts

print('Generating walk-forward visualization...')

COLORS = ['#2196F3', '#4CAF50', '#FF9800']

fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('AlphaLOB Walk-Forward Backtest Results', fontsize=14, fontweight='bold')

# ── Top row: equity curves ─────────────────────────────────────────────────
for w_idx, (returns, metrics) in enumerate(zip(all_returns, all_metrics)):
    ax = fig.add_subplot(gs[0, w_idx])
    if len(returns) > 0:
        equity = INITIAL_CAPITAL * np.cumprod(1.0 + returns)
        ax.plot(equity, color=COLORS[w_idx], linewidth=1.5, label='Equity')
        ax.axhline(INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.5, label='Initial capital')
        ax.fill_between(range(len(equity)), INITIAL_CAPITAL, equity,
                        where=equity >= INITIAL_CAPITAL, alpha=0.15, color='green')
        ax.fill_between(range(len(equity)), INITIAL_CAPITAL, equity,
                        where=equity < INITIAL_CAPITAL, alpha=0.15, color='red')
    name     = metrics['window']
    sharpe   = metrics['sharpe_ratio']
    max_dd   = metrics['max_drawdown']
    dir_acc  = metrics['directional_acc_30s']
    n_trades = int(metrics['total_trades'])
    ax.set_title(f'{name}\nSharpe={sharpe:.2f} | DD={max_dd*100:.1f}% | Acc={dir_acc*100:.1f}%', fontsize=9)
    ax.set_xlabel('Trades')
    ax.set_ylabel('Capital ($)')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'${v:,.0f}'))

# ── Bottom left: Sharpe comparison ───────────────────────────────────────────
ax_sharpe = fig.add_subplot(gs[1, 0])
names     = [m['window'] for m in all_metrics]
sharpes   = [m['sharpe_ratio'] for m in all_metrics]
bars = ax_sharpe.bar(names, sharpes, color=COLORS, edgecolor='white', linewidth=0.5)
ax_sharpe.axhline(2.3,              color='red',   linestyle='--', alpha=0.8, label='Target (2.3)', linewidth=1.5)
ax_sharpe.axhline(float(np.nanmean(sharpes)), color='gold', linestyle='-', alpha=0.9,
                   label=f'Mean ({np.nanmean(sharpes):.2f})', linewidth=2)
for bar, v in zip(bars, sharpes):
    ax_sharpe.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_sharpe.set_title('Sharpe Ratio by Window')
ax_sharpe.set_ylabel('Sharpe Ratio')
ax_sharpe.legend(fontsize=8)
ax_sharpe.grid(alpha=0.3, axis='y')

# ── Bottom middle: Directional accuracy ──────────────────────────────────────
ax_acc = fig.add_subplot(gs[1, 1])
accs   = [m['directional_acc_30s'] * 100 for m in all_metrics]
bars2  = ax_acc.bar(names, accs, color=COLORS, edgecolor='white', linewidth=0.5)
ax_acc.axhline(50.0,  color='red',    linestyle='--', alpha=0.8, label='Random (50%)', linewidth=1.5)
ax_acc.axhline(58.2,  color='orange', linestyle='--', alpha=0.8, label='Target (58.2%)', linewidth=1.5)
ax_acc.axhline(float(np.nanmean(accs)), color='gold', linestyle='-', alpha=0.9,
               label=f'Mean ({np.nanmean(accs):.1f}%)', linewidth=2)
for bar, v in zip(bars2, accs):
    ax_acc.text(bar.get_x() + bar.get_width()/2, v + 0.3,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_acc.set_title('30s Directional Accuracy')
ax_acc.set_ylabel('Accuracy (%)')
ax_acc.set_ylim(40, max(70, max(accs) + 5))
ax_acc.legend(fontsize=8)
ax_acc.grid(alpha=0.3, axis='y')

# ── Bottom right: Break-even cost vs target ──────────────────────────────────
ax_cost = fig.add_subplot(gs[1, 2])
be_costs = [m['break_even_bps'] for m in all_metrics]
bars3    = ax_cost.bar(names, be_costs, color=COLORS, edgecolor='white', linewidth=0.5)
ax_cost.axhline(SLIPPAGE_MAX_BPS, color='red',  linestyle='--', alpha=0.8,
                label=f'Max slippage ({SLIPPAGE_MAX_BPS} bps)', linewidth=1.5)
ax_cost.axhline(8.2, color='gold', linestyle='--', alpha=0.8, label='Target (8.2 bps)', linewidth=1.5)
for bar, v in zip(bars3, be_costs):
    ax_cost.text(bar.get_x() + bar.get_width()/2, v + 0.05,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_cost.set_title('Break-Even Transaction Cost (bps)')
ax_cost.set_ylabel('Break-even (bps)')
ax_cost.legend(fontsize=8)
ax_cost.grid(alpha=0.3, axis='y')

plt.savefig('/content/walkforward_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Walk-forward results chart saved to /content/walkforward_results.png')


In [ ]:
# Cell 10: Final OOS evaluation — the NEVER-TOUCHED 80%–100% slice
#
# IMPORTANT RULES FOR THIS CELL:
#   1. Run this cell EXACTLY ONCE after you are fully satisfied with walk-forward results.
#   2. Running it a second time = data snooping (your mental model has seen the result).
#   3. If you make ANY change to the model or strategy after seeing this result,
#      you MUST discard the Final OOS result and get new data.
#   4. This cell does NOT modify any model parameters — it is PURELY evaluative.
#
# Point-in-time vol: uses ALL data from [0%, 80%] (the max training window).

print('=' * 62)
print('  Cell 10: FINAL OUT-OF-SAMPLE EVALUATION (80%–100%)')
print('=' * 62)
print()
print('  ⚠️  This data was NEVER touched during model selection.')
print('  ⚠️  Running this cell a second time is data snooping!')
print()

final_oos_start_idx = int(n_total * FINAL_OOS_START)
df_final_oos        = df.slice(final_oos_start_idx, n_total - final_oos_start_idx)

print(f'  Final OOS slice: [{final_oos_start_idx:,}, {n_total:,}) = {len(df_final_oos):,} ticks')
print(f'  ({(1 - FINAL_OOS_START)*100:.0f}% of total data)')

# Point-in-time vol from [0%, 80%] training history
train_all_mid  = df.slice(0, final_oos_start_idx)['mid_price'].to_numpy()
train_all_lret = np.diff(np.log(np.maximum(train_all_mid, 1e-6)))
train_all_lret = train_all_lret[np.isfinite(train_all_lret)]
TICKS_PER_DAY  = 10 * 3600 * 24
final_daily_vol = float(np.std(train_all_lret)) * np.sqrt(TICKS_PER_DAY)
print(f'  Point-in-time daily vol (from 0–80%): {final_daily_vol*100:.4f}%')
print()

final_metrics, final_returns = run_backtest_on_window(
    df_final_oos, final_daily_vol, 'Final OOS'
)

print()
print('  === FINAL OOS RESULTS ===')
print(f'  Sharpe Ratio:          {final_metrics["sharpe_ratio"]:.3f}')
print(f'  Sortino Ratio:         {final_metrics["sortino_ratio"]:.3f}')
print(f'  Calmar Ratio:          {final_metrics["calmar_ratio"]:.3f}')
print(f'  Max Drawdown:          {final_metrics["max_drawdown"]*100:.2f}%')
print(f'  30s Dir. Accuracy:     {final_metrics["directional_acc_30s"]*100:.2f}%')
print(f'  Break-Even Cost (bps): {final_metrics["break_even_bps"]:.2f}')
print(f'  Total Return (OOS):    {final_metrics["total_return"]*100:.2f}%')
print(f'  Total Trades:          {final_metrics["total_trades"]:,}')

# Plot final OOS equity curve
if len(final_returns) > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    equity = INITIAL_CAPITAL * np.cumprod(1.0 + final_returns)
    ax.plot(equity, color='#9C27B0', linewidth=2, label='Final OOS Equity')
    ax.axhline(INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.6)
    ax.fill_between(range(len(equity)), INITIAL_CAPITAL, equity,
                    where=equity >= INITIAL_CAPITAL, alpha=0.15, color='green')
    ax.fill_between(range(len(equity)), INITIAL_CAPITAL, equity,
                    where=equity < INITIAL_CAPITAL, alpha=0.15, color='red')
    ax.set_title(f'Final OOS Equity Curve [80%–100%] | Sharpe={final_metrics["sharpe_ratio"]:.2f} | Acc={final_metrics["directional_acc_30s"]*100:.1f}%')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Capital ($)')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'${v:,.0f}'))
    plt.tight_layout()
    plt.savefig('/content/final_oos_equity.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Final OOS equity curve saved to /content/final_oos_equity.png')

print()
print('=' * 62)
print('  NOTEBOOK 05 COMPLETE — Walk-Forward Backtest')
print('=' * 62)
print('  Next step → Run 06_export_and_deploy.ipynb')
print('=' * 62)


In [ ]:
# Cell 11: Save backtest results to Drive + summary report

import json

print('Saving backtest results to Drive...')
os.makedirs('/content/drive/MyDrive/AlphaLOB', exist_ok=True)

# Compile summary dict (JSON-serializable)
def safe(v):
    """Convert numpy scalars and nan to Python float or None."""
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return None if not np.isfinite(v) else float(v)
    return v

summary = {
    'walk_forward_windows': [
        {k: safe(v) for k, v in m.items()} for m in all_metrics
    ],
    'final_oos': {k: safe(v) for k, v in final_metrics.items()},
    'mean_sharpe':       safe(np.nanmean([m['sharpe_ratio'] for m in all_metrics])),
    'mean_sortino':      safe(np.nanmean([m['sortino_ratio'] for m in all_metrics])),
    'mean_dir_acc_30s':  safe(np.nanmean([m['directional_acc_30s'] for m in all_metrics])),
    'mean_break_even_bps': safe(np.nanmean([m['break_even_bps'] for m in all_metrics])),
    'mean_max_drawdown': safe(np.nanmean([m['max_drawdown'] for m in all_metrics])),
}

results_path = '/content/drive/MyDrive/AlphaLOB/backtest_results.json'
with open(results_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ Results saved to {results_path}')

# Copy charts to Drive
import shutil
for chart in ['/content/walkforward_results.png', '/content/final_oos_equity.png']:
    if os.path.exists(chart):
        dst = f'/content/drive/MyDrive/AlphaLOB/{os.path.basename(chart)}'
        shutil.copy(chart, dst)
        print(f'✅ Chart saved to {dst}')

print()
print('=' * 62)
print('  FULL SUMMARY')
print('=' * 62)
print(f'  Mean Walk-Forward Sharpe:    {summary["mean_sharpe"]:.3f}  (target: 2.3)')
print(f'  Mean Walk-Forward Sortino:   {summary["mean_sortino"]:.3f}  (target: 3.1)')
print(f'  Mean 30s Dir. Accuracy:      {summary["mean_dir_acc_30s"]*100:.2f}%  (target: 58.2%)')
print(f'  Mean Break-even Cost:        {summary["mean_break_even_bps"]:.2f} bps  (target: 8.2)')
print(f'  Mean Max Drawdown:           {summary["mean_max_drawdown"]*100:.2f}%')
print(f'  Final OOS Sharpe:            {summary["final_oos"]["sharpe_ratio"]:.3f}')
print(f'  Final OOS 30s Accuracy:      {summary["final_oos"]["directional_acc_30s"]*100:.2f}%')
print()
print('  Next step → Run 06_export_and_deploy.ipynb')
print('=' * 62)
